#  Simple Regression with PyTorch
  Topic: Predicting Flood Water Depth from Rainfall Intensity

Goal: Train a small neural network that learns the relationship between rainfall intensity (mm/hr) and flood water depth (cm).

Steps:
- Generate synthetic data
- Build a Dataset and DataLoader
- Define a neural network model
- Train with a loss function and optimizer
- Evaluate and visualize results

In [ ]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Fix random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# ── 1. Generate Synthetic Data ────────────────────────────────────────────────
#
# True relationship (unknown to the model):
#   depth = 0.8 * rainfall^1.2 + noise
#
# This mimics a nonlinear rainfall-runoff response.

N = 500                                         # number of samples
rainfall = np.random.uniform(0, 50, N)          # rainfall intensity  [mm/hr]
noise    = np.random.normal(0, 2, N)            # measurement noise   [cm]
depth    = 0.8 * rainfall**1.2 + noise          # flood water depth   [cm]

In [ ]:
plt.plot(rainfall)
plt.ylabel("Rainfall Intensity (mm/hr)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Train / validation / test split  (60 / 20 / 20)
# reshape to (N, 1) for PyTorch compatibility
split = <...>
X_train = rainfall[<...>].astype(np.float32).reshape(-1, 1)
y_train = depth[<...>].astype(np.float32).reshape(-1, 1)

X_val   = rainfall[<...>].astype(np.float32).reshape(-1, 1)
y_val   = depth[<...>].astype(np.float32).reshape(-1, 1)

X_test  = rainfall[<...>].astype(np.float32).reshape(-1, 1)
y_test  = depth[<...>].astype(np.float32).reshape(-1, 1)

print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_val)}")
print(f"Testing  samples   : {len(X_test)}")

In [ ]:
# Normalize inputs to [0, 1] — always good practice!
<...>
X_train = <...>
X_val   = <...>
X_test  = <...>

# Normalize targets too (helps training stability)
<...>
y_train = <...>
y_val   = <...>
y_test  = <...>

In [ ]:
# ── 2. Build a PyTorch Dataset and DataLoader ─────────────────────────────────
#
# PyTorch wants data wrapped in a Dataset object so it can shuffle,
# batch, and feed samples to the model automatically.

class FloodDataset(Dataset):
    """Pairs of (rainfall, depth) observations."""

    def __init__(self, X, y):
        # Convert numpy arrays → PyTorch tensors
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)                  # total number of samples

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]    # one (input, label) pair

train_dataset = FloodDataset(X_train, y_train)
val_dataset   = FloodDataset(X_val,   y_val)
test_dataset  = FloodDataset(X_test,  y_test)

# DataLoader handles batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=<...>, shuffle=<...>)
val_loader   = DataLoader(val_dataset,   batch_size=<...>, shuffle=<...>)
test_loader  = DataLoader(test_dataset,  batch_size=<...>, shuffle=<...>)

print(f"Number of training batches  : {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of testing batches   : {len(test_loader)}")


In [ ]:
# ── 3. Define the Neural Network ──────────────────────────────────────────────

class RainfallDepthNet(nn.Module):
    def __init__(self):
        super().__init__()
        <...>

    def forward(self, x):
        <...>

model = RainfallDepthNet()
print(f"\nModel architecture:\n{model}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {sum(p.numel() for p in model.state_dict().values())}")
print(f"Total trainable parameters: {total_params}")


In [ ]:
# ── 4. Loss Function and Optimizer ────────────────────────────────────────────
#
# Loss     : Mean Squared Error — penalizes large prediction errors
# Optimizer: Adam — adapts the learning rate automatically

criterion = <...>
optimizer = <...>

In [ ]:
# ── 5. Training Loop ──────────────────────────────────────────────────────────

EPOCHS = <...>
train_losses = []
val_losses   = []

for epoch in range(1, EPOCHS + 1):

    # ── Training phase ────────────────────────────────────────────────────────
    <...>                           # tell PyTorch we are training
    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        <...>  # 1. clear old gradients
        prediction = <...>  # 2. forward pass
        loss = <...>  # 3. compute loss
        <...>  # 4. backpropagation
        <...>  # 5. update weights

        running_loss += loss.item() * len(X_batch)

    train_losses.append(running_loss / len(train_dataset))

    # ── Evaluation phase ──────────────────────────────────────────────────────
    <...>                            # tell PyTorch we are evaluating
    with torch.no_grad():                   # no gradient tracking needed
        val_loss = 0.0
        for X_batch, y_batch in val_loader:
            preds = <...>
            val_loss += criterion(preds, y_batch).item() * len(X_batch)
        val_losses.append(val_loss / len(val_dataset))

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}/{EPOCHS}  |  "
              f"Train Loss: {train_losses[-1]:.4f}  |  "
              f"Val Loss:  {val_losses[-1]:.4f}")

In [ ]:
# Loss curves
plt.plot(train_losses, label="Train loss", color="#1f77b4")
plt.plot(val_losses,  label="Validation loss",  color="#ff7f0e", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(alpha=0.3)


In [ ]:
preds = []
model.eval()
with torch.no_grad():
    test_loss = 0.0
    for X_batch, y_batch in test_loader:
        pred = model(X_batch)  # shape: (batch_size, 1)

        # move to CPU, remove last dim, convert to 1D numpy
        preds.append(pred.squeeze(-1).cpu().numpy())

        test_loss += criterion(pred, y_batch).item() * len(X_batch)

    test_loss /= len(test_dataset)

# Final shape: (num_test_samples,)
preds = np.concatenate(preds, axis=0)

y_pred = preds  * (depth_max - depth_min) + depth_min
y_gt  = y_test * (depth_max - depth_min) + depth_min

In [ ]:
# Scatter plot of predictions vs. ground truth
plt.plot(y_gt, y_pred, '*')
plt.plot([depth_min, depth_max], [depth_min, depth_max], color="r", linestyle="--")
plt.xlabel("Ground truth (test)")
plt.ylabel("Predictions")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# -- 7 Evaluation Metrics ───────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_gt, y_pred)
r2  = r2_score(y_gt, y_pred)
print(f"Test MSE: {mse:.4f}")
print(f"Test R² : {r2:.4f}")

In [ ]:
# Nash-Sutcliffe Efficiency (NSE)
y_obs = np.asarray(y_gt).reshape(-1)
y_sim = np.asarray(y_pred).reshape(-1)

den = np.sum((y_obs - np.mean(y_obs)) ** 2)
num = np.sum((y_obs - y_sim) ** 2)

nse = 1 - (num / den) if den != 0 else np.nan
print(f"Test NSE: {nse:.4f}")